# Figure 2A Monocle3 trajectories

Reproduce four independent views: cell annotation label/legend and embryonic stage label/legend. Original Ia and II are displayed separately as `Ia candidate` and `II candidate`; their shared `Ia/II precursor candidate` and all other accepted labels are retained. All four views use exactly the same cells, original UMAP, principal graph, E9.5 root, pseudotime and point order. Display labels do not impose lineage endpoints.

Run all numbered cells from the project root. The default reuses the selected RDS/H5AD pair. If the pair is absent, or `RECOMPUTE=True`, cell 2 runs the selected Monocle3 analysis from the saved integrated atlas, using original counts, UMAP, fixed parameters and seed 42. This starts from the saved integration stage, without repeating upstream QC/integration. The R runtime must provide Monocle3 1.4.27; `RSCRIPT` and `R_LIBS_USER` may override the bundled project runtime.

- Input: `ROOT`.
- Reusable pair: `ROOT` and `Monocle3-1.h5ad`.
- Figures: `ROOT`.
- PNG: 600 DPI. PDF: semantic Arial text and vector geometry in three sibling clipping groups.

The H5AD is a lightweight trajectory object and links to original expression data; the native RDS also retains the selected count matrix. Changing appearance requires only cells 3–6; changing analysis parameters requires rerunning cell 2 with `RECOMPUTE=True`.


In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import os
import sys
import tempfile

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
if ROOT.name == 'Code_def': ROOT = ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
INPUT_FILE = ROOT
# Set the file or directory path.
RESULT_DIR = ROOT
# Uses Monocle3-1.h5ad generated by the Monocle3 fitting step.
RESULT_FILE = ROOT
# Uses Monocle3-1.rds generated by the Monocle3 fitting step.
MODEL_FILE = ROOT
# Set the file or directory path.
FIGURES = ROOT
# Set the executable or tool directory path.
R_SCRIPT = ROOT
# Set the executable or tool directory path.
R_LIBRARY = ROOT
RANDOM_SEED, N_THREADS = 42, 4
# Set the temporary parent directory; this run gets its own removable folder.
TASK_TMP = Path(tempfile.mkdtemp(prefix="monocle3-", dir=ROOT))
TASK_TMP.mkdir(parents=True, exist_ok=True)
# Set the file or directory path.
os.environ['MPLCONFIGDIR'] = str(ROOT)
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):
    os.environ[key] = str(N_THREADS)
sys.dont_write_bytecode = True

import io
import json
import gzip
import hashlib
import subprocess
import shutil
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.io import mmwrite
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib import font_manager, colors as mcolors, markers as mmarkers
from matplotlib.path import Path as MplPath
from matplotlib.text import Text
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from pypdf import PdfReader
from pypdf.generic import ContentStream
from PIL import Image
from IPython.display import display

ad.settings.allow_write_nullable_strings = True
plt.rcdefaults()
# Set the file or directory path.
ARIAL_FILE = ROOT
font_manager.fontManager.addfont(str(ARIAL_FILE))
plt.rcParams.update({'font.family':'Arial', 'figure.facecolor':'white',
                     'axes.grid':False, 'axes.unicode_minus':False})
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

GROUP_MAP = {'Type Ia + Type II precursor candidate': 'Ia/II precursor candidate', 'Immature Type Ia SGN': 'Ia candidate', 'Immature Type II SGN': 'II candidate', 'Type Ib + Type Ic precursor candidate': 'Ib/Ic precursor candidate', 'Emerging Type Ic': 'Ib/Ic precursor candidate', 'Immature Type Ib SGN': 'Ib/Ic candidate', 'Immature Type Ic SGN': 'Ib/Ic candidate'}
LABEL_TEXT = {'CVG neuroblast': 'CVG neuroblast', 'O-2: delaminating otic/CVG neuroblast': 'Delaminating\notic/CVG', 'O-3: developing CVG neuron (not SGN-specific)': 'Developing CVG', 'Undifferentiated / proliferative CVG': 'Proliferative CVG', 'Late CVG': 'Late CVG', 'Differentiating SGN': 'Differentiating SGN', 'Undifferentiated SGN': 'Undifferentiated\nSGN', 'Ia/II precursor candidate': 'Ia/II precursor\ncandidate', 'Ia candidate': 'Ia candidate', 'II candidate': 'II candidate', 'Ib/Ic precursor candidate': 'Ib/Ic precursor\ncandidate', 'Ib/Ic candidate': 'Ib/Ic candidate'}
STATE_COLORS = {'CVG neuroblast': '#A26C4A', 'O-2: delaminating otic/CVG neuroblast': '#277E75', 'O-3: developing CVG neuron (not SGN-specific)': '#C5849C', 'Undifferentiated / proliferative CVG': '#9967AB', 'Late CVG': '#8B8C48', 'Differentiating SGN': '#C8924B', 'Undifferentiated SGN': '#8D9B9E', 'Ia/II precursor candidate': '#C68472', 'Ia candidate': '#8F3E35', 'II candidate': '#4F7851', 'Ib/Ic precursor candidate': '#7A9FBB', 'Ib/Ic candidate': '#2E5B8D'}
LABEL_LAYOUT = {'CVG neuroblast': ((16.5, 2.5), (16.8, 0.7)), 'O-2: delaminating otic/CVG neuroblast': ((13.7, 1.5), (16.3, 0.8)), 'O-3: developing CVG neuron (not SGN-specific)': ((14.0, -3.1), (13.4, -2.6)), 'Undifferentiated / proliferative CVG': ((17.0, -0.3), (16.8, 0)), 'Late CVG': ((17.0, -1.5), (15.0, -1.0)), 'Differentiating SGN': ((9.0, -5.7), (9.2, -4.0)), 'Undifferentiated SGN': ((8.0, 0), (4.0, -0.5)), 'Ia/II precursor candidate': ((6.0, 8.7), (0.84, 8.88)), 'Ia candidate': ((-1.5, 13.6), (-1.5, 12.0)), 'II candidate': ((5.7, 11.6), (3.9290938, 10.0741920)), 'Ib/Ic precursor candidate': ((-4.8, 2.0), (-2.2, 3.2)), 'Ib/Ic candidate': ((-5.2, 6.8), (-3.15, 5.56))}

STAGE_COLORS = {'E9.5': '#D157A0', 'E10.5': '#CC0744', 'E11.5': '#3B9700', 'E13.5': '#3B5DFF', 'E14.5': '#D16100', 'E15.5': '#00846F', 'E16.5': '#BC23FF', 'E17.5': '#6A3A4C', 'E18.5': '#9B9700'}
STAGE_LAYOUT = {'E9.5': ((16.8, 2.35), None), 'E10.5': ((14.25, 1.1), None), 'E11.5': ((18.0, -1.1), None), 'E13.5': ((9.1, -5.7), (9.3, -3.9)), 'E14.5': ((6.8, -1.1), None), 'E15.5': ((5.95, 3.1), (4.0, 2.3)), 'E16.5': ((3.3, 7.6), None), 'E17.5': ((-3.2, 8.3), None), 'E18.5': ((-1.5, 13.6), (-1.5, 12.0))}
FIGURE_MANIFEST = {
    'version':1, 'code_file':'fig2_moncle3-1.ipynb',
    'result_file':RESULT_FILE.name, 'model_file':MODEL_FILE.name,
    'figures_directory':str(FIGURES.relative_to(ROOT)),
    'figure_files':[f'Monocle3-{view}-{mode}-1.{extension}'
                    for view in ('cell','stage') for mode in ('label','legend')
                    for extension in ('png','pdf')],
    'group_map':GROUP_MAP, 'display_colors':STATE_COLORS, 'stage_colors':STAGE_COLORS,
    'cell_label_text':LABEL_TEXT, 'cell_label_layout':LABEL_LAYOUT, 'stage_label_layout':STAGE_LAYOUT,
    'main_fork_marker':'Y_47', 'point_order_seed':RANDOM_SEED,
    'trajectory_changed':False,
    'annotation_note':'Separate Ia and II candidate labels; retain shared Ia/II precursor and accepted Ib/Ic grouping.'
}


In [ ]:
#2. =========== Fit or load the selected Monocle3 model
RECOMPUTE = False  # Set True to refit the model from the saved integration data.
params = {'seed': RANDOM_SEED, 'threads': N_THREADS, 'ncenter': 200, 'cluster_k': 20,
          'graph_neighbors': 25, 'minimal_branch_len': 10, 'use_partition': True,
          'close_loop': False, 'prune_graph': True, 'partition_qval': 0.05,
          'cluster_method': 'leiden', 'nn_method': 'annoy', 'nn_metric': 'euclidean',
          'root_strategy': 'modal principal node of earliest-stage cells',
          'exclude_mitochondrial': True}
# Set the paths used by the embedded R step.
R_SOURCE = r'''ROOT <- "."  # Set the file paths before running this R step.
suppressPackageStartupMessages({
  library(monocle3)
  library(Matrix)
  library(SingleCellExperiment)
  library(igraph)
  library(jsonlite)
})
args <- commandArgs(trailingOnly=TRUE)
work <- normalizePath(args[1], mustWork=TRUE)
setwd(work)
# Uses parameters.json generated by the trajectory input preparation step.
params <- fromJSON(ROOT)
set.seed(params$seed)
options(mc.cores=params$threads)
# Uses cells.csv generated by the Monocle3 input preparation step.
cells <- read.csv(ROOT,row.names=1,check.names=FALSE,stringsAsFactors=FALSE)
# Uses genes.csv generated by the Monocle3 input preparation step.
genes <- read.csv(ROOT,row.names=1,check.names=FALSE,stringsAsFactors=FALSE)
# Uses umap.csv generated by the trajectory input preparation step.
xy <- as.matrix(read.csv(ROOT,row.names=1,check.names=FALSE))
# Uses counts.mtx.gz generated by the Monocle3 input preparation step.
con <- gzfile(ROOT,'rt')
counts <- as(readMM(con),'CsparseMatrix')
close(con)
rownames(counts) <- rownames(genes)
colnames(counts) <- rownames(cells)
genes_keep <- if (params$exclude_mitochondrial) !grepl('^mt-',genes$gene_short_name,ignore.case=TRUE) else rep(TRUE,nrow(genes))
counts <- counts[genes_keep,,drop=FALSE]
genes <- genes[genes_keep,,drop=FALSE]
cells$trajectory_eligible <- TRUE
cds <- new_cell_data_set(counts,cell_metadata=cells,gene_metadata=genes)
reducedDims(cds)$UMAP <- xy
cds <- cluster_cells(cds,reduction_method='UMAP',k=params$cluster_k,
                     random_seed=params$seed,cluster_method=params$cluster_method,
                     partition_qval=params$partition_qval,
                     nn_control=list(method=params$nn_method,metric=params$nn_metric,cores=params$threads))
cds <- learn_graph(cds,use_partition=params$use_partition,close_loop=params$close_loop,
                   learn_graph_control=list(ncenter=params$ncenter,
                                            minimal_branch_len=params$minimal_branch_len,
                                            prune_graph=params$prune_graph,
                                            nn.k=params$graph_neighbors,nn.cores=params$threads))
graph <- principal_graph(cds)$UMAP
coords <- t(principal_graph_aux(cds)$UMAP$dp_mst)
closest <- principal_graph_aux(cds)$UMAP$pr_graph_cell_proj_closest_vertex
closest_name <- rownames(coords)[as.integer(as.matrix(closest[colnames(cds),,drop=FALSE])[,1])]
component <- components(graph)$membership
cell_component <- unname(component[closest_name])
early <- which(cells$stage_numeric==min(cells$stage_numeric))
root_nodes <- names(sort(table(closest_name[early]),decreasing=TRUE))[1]
cds <- order_cells(cds,root_pr_nodes=root_nodes)
pt <- unname(pseudotime(cds)[colnames(cds)])
assignments <- data.frame(cell_id=colnames(cds),pseudotime=pt,
                          partition=as.character(partitions(cds)),
                          component=cell_component,nearest_node=closest_name,
                          monocle3_cluster=as.character(monocle3::clusters(cds)[colnames(cds)]))
nodes <- data.frame(node=rownames(coords),UMAP1=coords[,1],UMAP2=coords[,2],
                    degree=unname(degree(graph)[rownames(coords)]),
                    component=unname(component[rownames(coords)]),root=rownames(coords)%in%root_nodes)
edges <- as_data_frame(graph,what='edges')
roots <- data.frame(component=unique(cell_component[early]),root_node=root_nodes,
                    earliest_stage=min(cells$stage_numeric),earliest_n=length(early),
                    component_n=ncol(cds),root_context=paste(sort(unique(cells$cell_type_plot[early])),collapse='; '))
software <- list(R=as.character(getRversion()),monocle3=as.character(packageVersion('monocle3')),
                 igraph=as.character(packageVersion('igraph')))
metadata(cds)$monocle3_pipeline <- list(parameters=params,software=software,
                                      # Uses input_file.txt generated by the Monocle3 input preparation step.
                                      input_file=readLines(ROOT),
                                      # Uses input_sha256.txt generated by the Monocle3 input preparation step.
                                      input_sha256=readLines(ROOT),
                                      root_nodes=root_nodes)
# Uses figure_manifest.json generated by the Monocle3 input preparation step.
metadata(cds)$monocle3_fig2A <- fromJSON(ROOT)
# Uses Monocle3.rds generated by the Monocle3 R fitting step.
base::saveRDS(cds,ROOT,compress='gzip')
# Uses main_cells.csv generated by the Monocle3 R fitting step.
write.csv(assignments,ROOT,row.names=FALSE)
# Uses main_nodes.csv generated by the Monocle3 R fitting step.
write.csv(nodes,ROOT,row.names=FALSE)
# Uses main_edges.csv generated by the Monocle3 R fitting step.
write.csv(edges,ROOT,row.names=FALSE)
# Uses main_roots.csv generated by the Monocle3 R fitting step.
write.csv(roots,ROOT,row.names=FALSE)
# Uses software.json generated by the Monocle3 R fitting step.
write_json(software,ROOT,auto_unbox=TRUE)
cat('Monocle3:',ncol(cds),'cells,',sum(is.finite(pt)),'ordered,',vcount(graph),'nodes,',ecount(graph),'edges; root',root_nodes,'\n')
'''

with INPUT_FILE.open('rb') as stream:
    input_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
if RECOMPUTE or not (RESULT_FILE.is_file() and MODEL_FILE.is_file()):
    with tempfile.TemporaryDirectory(prefix='run-', dir=TASK_TMP) as workdir:
        WORK = Path(workdir)
        a = ad.read_h5ad(INPUT_FILE)
        original_umap = np.asarray(a.obsm['X_umap']).copy()
        cm = a.obs[['stage_standardized', 'cell_type_plot', 'stage_celltype', 'study_id',
                    'library_id', 'biological_replicate_id', 'cochlear_region',
                    'source_state_evaluation_allowed', 'source_rna_mixture_review_flag', 'leiden']].copy()
        cm['display_state'] = a.obs.cell_type_plot.astype(str).replace(GROUP_MAP)
        cm.index.name = 'cell_id'
        cm['stage_numeric'] = cm.stage_standardized.astype(str).str[1:].astype(float)
        cm['trajectory_eligible'] = True
        # Uses cells.csv generated by the Monocle3 input preparation step.
        cm.to_csv(ROOT)
        gm = a.var[['gene_symbol']].rename(columns={'gene_symbol': 'gene_short_name'}).copy()
        gm.index.name = 'gene_id'
        # Uses genes.csv generated by the Monocle3 input preparation step.
        gm.to_csv(ROOT)
        # Uses umap.csv generated by the trajectory input preparation step.
        pd.DataFrame(original_umap, index=cm.index, columns=['UMAP1', 'UMAP2']).to_csv(ROOT)
        # Uses counts.mtx.gz generated by the Monocle3 input preparation step.
        with gzip.open(ROOT, 'wb', compresslevel=3) as f:
            mmwrite(f, a.layers['counts'].T, field='integer')
        # Uses parameters.json generated by the trajectory input preparation step.
        (ROOT).write_text(json.dumps(params, sort_keys=True), encoding='utf8')
        # Uses figure_manifest.json generated by the Monocle3 input preparation step.
        (ROOT).write_text(json.dumps(FIGURE_MANIFEST), encoding='utf8')
        # Uses input_sha256.txt generated by the Monocle3 input preparation step.
        (ROOT).write_text(input_sha+'\n', encoding='utf8')
        # Uses input_file.txt generated by the Monocle3 input preparation step.
        (ROOT).write_text(str(INPUT_FILE.relative_to(ROOT))+'\n', encoding='utf8')
        # Set the file or directory path.
        (ROOT).write_text(R_SOURCE, encoding='utf8')
        env = os.environ.copy()
        env.update({'R_LIBS_USER': str(R_LIBRARY), 'TMP': str(WORK), 'TEMP': str(WORK),
                    'LC_ALL': 'English_United States.utf8'})
        # Set the file or directory path.
        completed = subprocess.run([str(R_SCRIPT), '--vanilla', str(ROOT), str(WORK)],
                                   capture_output=True, text=True, encoding='utf8', errors='replace',
                                   env=env, timeout=7200)
        if completed.returncode:
            raise RuntimeError(completed.stdout[-3000:]+'\n'+completed.stderr[-3000:])
        if completed.stderr.strip(): print(completed.stderr.strip())
        # Uses main_cells.csv generated by the Monocle3 R fitting step.
        trajectory = pd.read_csv(ROOT, index_col='cell_id').reindex(a.obs_names)
        # Uses main_nodes.csv generated by the Monocle3 R fitting step.
        nodes = pd.read_csv(ROOT, index_col='node')
        # Uses main_edges.csv generated by the Monocle3 R fitting step.
        edges = pd.read_csv(ROOT)
        # Uses main_roots.csv generated by the Monocle3 R fitting step.
        roots = pd.read_csv(ROOT)
        obs_keys = ['cell_type_plot', 'stage_standardized', 'stage_celltype', 'leiden', 'study_id',
                    'library_id', 'biological_replicate_id', 'cochlear_region', 'GSM', 'original_cell_id',
                    'source_cell_id', 'original_annotation', 'integration_batch', 'source_review_flag',
                    'source_rna_mixture_review_flag', 'source_state_evaluation_allowed', 'author_SGN_tree_member']
        obs = a.obs[[x for x in obs_keys if x in a.obs]].copy().join(trajectory)
        result = ad.AnnData(X=sparse.csr_matrix((a.n_obs, 0)), obs=obs)
        result.obsm['X_umap'] = original_umap
        result.obs['display_state'] = pd.Categorical(a.obs.cell_type_plot.astype(str).replace(GROUP_MAP), categories=list(STATE_COLORS), ordered=True)
        result.uns['monocle3'] = {'palettes_json': json.dumps({'display_state': STATE_COLORS, 'stage_standardized': STAGE_COLORS}, sort_keys=True), 'parameters_json': json.dumps(params, sort_keys=True),
                                 'input_sha256': input_sha, 'input_file': str(INPUT_FILE.relative_to(ROOT)),
                                 'model_file': MODEL_FILE.name, 'nodes': nodes, 'edges': edges, 'roots': roots,
                                 # Uses software.json generated by the Monocle3 R fitting step.
                                 'software_json': (ROOT).read_text(encoding='utf8'),
                                 'counts_location': 'input_file: layers/counts; native RDS also retains non-mitochondrial counts',
                                 'original_annotations_retained': True, 'all_cells_included': True}
        result.uns['monocle3_fig2A'] = {'manifest_json':json.dumps(FIGURE_MANIFEST,sort_keys=True)}
        # Uses Monocle3.rds generated by the Monocle3 R fitting step.
        with (ROOT).open('rb') as stream:
            result.uns['monocle3']['model_sha256'] = hashlib.file_digest(stream,'sha256').hexdigest()
        # Set the file or directory path.
        result.write_h5ad(ROOT, compression='gzip')
        # Set the file or directory path.
        shutil.copyfile(ROOT, RESULT_FILE)
        # Uses Monocle3.rds generated by the Monocle3 R fitting step.
        shutil.copyfile(ROOT, MODEL_FILE)
else:
    result = ad.read_h5ad(RESULT_FILE)
    saved = result.uns['monocle3']
    assert saved['input_sha256'] == input_sha and saved['parameters_json'] == json.dumps(params, sort_keys=True), 'Input or parameters changed; set RECOMPUTE=True.'
assert json.loads(result.uns['monocle3_fig2A']['manifest_json']) == json.loads(json.dumps(FIGURE_MANIFEST)), 'Figure settings changed; update the saved manifest before reuse.'
with MODEL_FILE.open('rb') as stream:
    assert result.uns['monocle3']['model_sha256'] == hashlib.file_digest(stream,'sha256').hexdigest(), 'RDS/H5AD mismatch.'
assert result.obs.display_state.astype(str).tolist() == result.obs.cell_type_plot.astype(str).replace(GROUP_MAP).tolist()
print(f'Monocle3 ready: {result.n_obs:,} cells, {len(result.uns["monocle3"]["nodes"])} nodes; selected saved pair verified.')


In [ ]:
#3. =========== Read the selected trajectory
result = ad.read_h5ad(RESULT_FILE)
saved = result.uns['monocle3']
params = json.loads(saved['parameters_json'])
xy = np.asarray(result.obsm['X_umap'])
nodes, edges, roots = saved['nodes'], saved['edges'], saved['roots']
values = result.obs.cell_type_plot.astype(str).replace(GROUP_MAP).to_numpy()
stage = result.obs.stage_standardized.astype(str).to_numpy()
assert sorted(set(stage),key=lambda value:float(value[1:])) == list(STAGE_COLORS)
root_node = roots.root_node.iloc[0]
MAIN_FORK = 'Y_47'  # A display marker on the selected fitted graph, not a fit constraint.
ANCHOR_CELL = 'GSE195500__GSE195500_L05__TGCCGAGTCACGGAGA-1_5'
order = np.random.default_rng(RANDOM_SEED).permutation(result.n_obs)
point_colors = np.array([STATE_COLORS[value] for value in values])
lo, hi = xy.min(axis=0), xy.max(axis=0)
pad = np.maximum((hi-lo)*.07, .3)
limits = [(lo[j]-pad[j], hi[j]+pad[j]) for j in range(2)]


In [ ]:
#4. =========== Editable PDF: three sibling vector groups
def editable_pdf(fig, output):
    """Keep paths and semantic text objects editable; no images or Form XObjects."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit = 72 / fig.dpi
    width, height = fig.get_size_inches() * 72
    # Set the file or directory path.
    for name, filename in [('Arial', str(ROOT)), ('ArialBold', str(ROOT))]:
        if name not in pdfmetrics.getRegisteredFontNames():
            # Set the file or directory path.
            pdfmetrics.registerFont(TTFont(name, str(ROOT)))
    data = io.BytesIO()
    pdf = rl_canvas.Canvas(data, pagesize=(width, height), pageCompression=1, invariant=1, initialFontName='Arial')
    pdf.setTitle(Path(output).stem)
    pdf.setCreator('Monocle3 vector export')

    def color(value, stroke=False, alpha=None):
        rgba = mcolors.to_rgba(value)
        (pdf.setStrokeColorRGB if stroke else pdf.setFillColorRGB)(*rgba[:3])
        (pdf.setStrokeAlpha if stroke else pdf.setFillAlpha)(rgba[3] if alpha is None else alpha)

    def path(vertices, codes=None):
        p = pdf.beginPath()
        previous = None
        for points, code in MplPath(vertices, codes).iter_segments(curves=True, simplify=False):
            v = points.reshape(-1, 2) * unit
            if code in (MplPath.CURVE3, MplPath.CURVE4) and previous is not None:
                direction, controls = v[-1] - previous, v[:-1] - previous
                crosses = controls[:, 0] * direction[1] - controls[:, 1] * direction[0]
                along = controls @ direction
                if np.allclose(crosses, 0, atol=1e-8) and np.all(along >= 0) and np.all(along <= direction @ direction):
                    v, code = v[-1:], MplPath.LINETO
            if code == MplPath.MOVETO:
                p.moveTo(*v[0])
            elif code == MplPath.LINETO:
                p.lineTo(*v[0])
            elif code == MplPath.CURVE3:
                q, endpoint = v
                c1 = previous + 2 / 3 * (q - previous)
                c2 = endpoint + 2 / 3 * (q - endpoint)
                p.curveTo(*c1, *c2, *endpoint)
            elif code == MplPath.CURVE4:
                p.curveTo(*v.ravel())
            elif code == MplPath.CLOSEPOLY:
                p.close()
            if len(v):
                previous = v[-1]
        return p

    def draw(artist):
        if not artist.get_visible():
            return
        pdf.saveState()
        if isinstance(artist, Text):
            if not artist.get_text():
                pdf.restoreState()
                return
            position = artist.get_transform().transform(artist.get_position())
            layout = artist._get_layout(renderer)[1]
            angle = artist.get_rotation()
            font = 'ArialBold' if artist.get_fontweight() in ('bold', 'heavy', 700, 800, 900) else 'Arial'
            color(artist.get_color())
            text = pdf.beginText()
            text.setFont(font, artist.get_fontsize())
            theta = np.deg2rad(angle)
            c, s = np.cos(theta), np.sin(theta)
            for line, _, x, y in layout:
                origin = (position + [x, y]) * unit
                text.setTextTransform(c, s, -s, c, *origin)
                text.textOut(line)
            pdf.drawText(text)
        elif isinstance(artist, PathCollection):
            centers = artist.get_offset_transform().transform(artist.get_offsets()) * unit
            sizes = artist.get_sizes()
            faces, edges = artist.get_facecolors(), artist.get_edgecolors()
            linewidths = artist.get_linewidths()
            for i, (x, y) in enumerate(centers):
                if not np.isfinite(x + y):
                    continue
                fill = len(faces) and faces[i % len(faces), 3] > 0
                stroke = len(edges) and edges[i % len(edges), 3] > 0 and linewidths[i % len(linewidths)] > 0
                if fill:
                    color(faces[i % len(faces)])
                if stroke:
                    color(edges[i % len(edges)], stroke=True)
                    pdf.setLineWidth(linewidths[i % len(linewidths)])
                marker = artist.get_paths()[i % len(artist.get_paths())]
                vertices = marker.vertices*np.sqrt(sizes[i % len(sizes)])/unit + np.array([x,y])/unit
                pdf.drawPath(path(vertices,marker.codes),stroke=int(stroke),fill=int(fill))
        elif isinstance(artist, Line2D):
            color(artist.get_color(), stroke=True, alpha=artist.get_alpha())
            pdf.setLineWidth(artist.get_linewidth())
            pdf.setLineJoin(1)
            points = artist.get_transform().transform(artist.get_xydata())
            if artist.get_linestyle() not in ('None', 'none', '', ' '):
                if artist.get_linestyle() in ('--', ':', '-.'):
                    pdf.setDash({'--': [3, 2], ':': [1, 1.5], '-.': [3, 2, 1, 2]}[artist.get_linestyle()])
                pdf.drawPath(path(points), stroke=1, fill=0)
            if artist.get_marker() not in ('None', 'none', '', ' ', None):
                marked = points
                every = artist.get_markevery()
                if isinstance(every, (int, np.integer)):
                    marked = points[::every]
                elif isinstance(every, tuple):
                    marked = points[every[0]::every[1]]
                elif every is not None:
                    marked = points[every]
                marker = mmarkers.MarkerStyle(artist.get_marker())
                mp = marker.get_path().transformed(marker.get_transform())
                scale = artist.get_markersize() / unit
                color(artist.get_markeredgecolor(), stroke=True)
                pdf.setLineWidth(artist.get_markeredgewidth())
                face = artist.get_markerfacecolor()
                fill = mcolors.to_rgba(face)[3] > 0
                if fill:
                    color(face)
                for point in marked:
                    if artist.get_marker() == 'o':
                        pdf.circle(*(point * unit), artist.get_markersize() / 2, stroke=1, fill=int(fill))
                    else:
                        pdf.drawPath(path(mp.vertices * scale + point, mp.codes), stroke=1, fill=int(fill))
        elif isinstance(artist, Patch):
            color(artist.get_edgecolor(), stroke=True)
            color(artist.get_facecolor())
            pdf.setLineWidth(artist.get_linewidth())
            p = artist.get_path().transformed(artist.get_transform())
            pdf.drawPath(path(p.vertices, p.codes), stroke=int(artist.get_linewidth() > 0 and artist.get_edgecolor()[3] > 0),
                         fill=int(artist.get_fill() and artist.get_facecolor()[3] > 0))
        pdf.restoreState()

    groups = {name: [] for name in ('Graphics', 'Axes/Guides', 'Text')}
    for ax in fig.axes:
        groups['Axes/Guides'].extend(ax.spines.values())
        for axis in (ax.xaxis, ax.yaxis):
            for tick in axis._update_ticks():
                groups['Axes/Guides'].extend((tick.tick1line,tick.tick2line,tick.gridline))
                groups['Text'].extend((tick.label1,tick.label2))
            groups['Text'].extend((axis.label,axis.get_offset_text()))
        groups['Text'].extend([ax.title,ax._left_title,ax._right_title,*ax.texts])
        groups['Graphics'].extend([*ax.collections,*ax.lines,*ax.patches])
        legend = ax.get_legend()
        if legend:
            groups['Text'].extend(legend.get_texts())
            groups['Graphics'].extend(legend.findobj(lambda a:isinstance(a,(PathCollection,Line2D,Patch))))
    groups['Text'].extend(fig.texts)
    for artists in groups.values():
        pdf.saveState()
        rectangle = pdf.beginPath()
        rectangle.rect(0,0,width,height)
        pdf.clipPath(rectangle,stroke=0,fill=0,fillMode=1)
        for artist in sorted({id(a):a for a in artists}.values(),key=lambda a:a.get_zorder()):
            draw(artist)
        pdf.restoreState()
    pdf.showPage()
    pdf.save()
    Path(output).write_bytes(data.getvalue())


In [ ]:
#5. =========== Cell annotation and stage label/legend views
POINT_SIZE, POINT_ALPHA = 1.35, .82
GRAPH_WIDTH, LABEL_SIZE = .85, 8
PNG_DPI = 600
FIGURE_TITLE = 'Monocle3 | original UMAP | ncenter=200, seed=42'

figure_checks = []
for view in ('cell','stage'):
    palette = STATE_COLORS if view=='cell' else STAGE_COLORS
    groups = values if view=='cell' else stage
    label_text = LABEL_TEXT if view=='cell' else {key:key for key in STAGE_COLORS}
    layout = LABEL_LAYOUT if view=='cell' else STAGE_LAYOUT
    colors = point_colors if view=='cell' else np.array([palette[value] for value in groups])
    for mode in ('label','legend'):
        fig = plt.figure(figsize=(8.6,8.3) if mode=='label' else (11.2,7.7))
        ax = fig.add_axes([.12,.11,.80,.80] if mode=='label' else [.08,.12,.54,.80])
        dots = ax.scatter(xy[order,0],xy[order,1],c=colors[order],s=POINT_SIZE,
                         alpha=POINT_ALPHA,linewidths=0,edgecolors='none')
        for source,target in edges[['from','to']].itertuples(index=False,name=None):
            segment = nodes.loc[[source,target],['UMAP1','UMAP2']].to_numpy()
            ax.plot(segment[:,0],segment[:,1],color='black',lw=GRAPH_WIDTH,zorder=3)
        ax.scatter(*nodes.loc[root_node,['UMAP1','UMAP2']].to_numpy(float),s=30,
                   facecolors='white',edgecolors='black',lw=.9,zorder=5)
        ax.scatter(*nodes.loc[MAIN_FORK,['UMAP1','UMAP2']].to_numpy(float),s=25,
                   c='#E7BD58',marker='D',edgecolors='black',lw=.5,zorder=5)
        ax.set(xlim=limits[0],ylim=limits[1],xlabel='UMAP1',ylabel='UMAP2')
        ax.set_box_aspect(1)
        title = FIGURE_TITLE if view=='cell' else 'Monocle3 | embryonic stage | ncenter=200, seed=42'
        ax.set_title(title,fontsize=9,pad=8)
        ax.tick_params(labelsize=7,width=1.1)
        ax.spines[['top','right']].set_visible(False)
        for side in ('left','bottom'): ax.spines[side].set_linewidth(1.1)
        labels, anchors = [], []
        if mode=='label':
            for group,text in label_text.items():
                position,target = layout[group]
                points = xy[groups==group]
                if target is None: target = np.median(points,axis=0)
                anchor = (xy[result.obs_names.get_loc(ANCHOR_CELL)] if view=='cell' and group=='Ib/Ic precursor candidate'
                          else points[np.argmin(np.linalg.norm(points-target,axis=1))])
                anchors.append(anchor)
                labels.append(ax.text(*position,text,ha='center',va='center',fontsize=LABEL_SIZE,
                    path_effects=[pe.withStroke(linewidth=1.6,foreground='white')],zorder=7))
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            boxes = []
            for text,anchor in zip(labels,anchors):
                box = text.get_window_extent(renderer)
                assert not any(box.padded(2).overlaps(old) for old in boxes)
                boxes.append(box)
                center = ax.transData.transform(text.get_position())
                anchor_px = ax.transData.transform(anchor)
                delta = anchor_px-center
                if np.linalg.norm(delta)>12:
                    scale = np.array([box.width/2,box.height/2])/np.maximum(np.abs(delta),1e-8)
                    edge = center+delta*min(scale)
                    line = ax.transData.inverted().transform(np.stack([edge,anchor_px]))
                    ax.plot(line[:,0],line[:,1],color='.32',lw=.42,alpha=.70,zorder=6)
        else:
            handles = [Line2D([],[],marker='o',linestyle='',color=color,markersize=5,
                label=f'{label_text[group].replace(chr(10)," ")} ({np.sum(groups==group):,})')
                for group,color in palette.items()]
            ax.legend(handles=handles,loc='upper left',bbox_to_anchor=(1.03,1),
                      frameon=False,fontsize=7,labelspacing=.65)
        fig.text(.12,.030,'7,355 cells | open circle: E9.5 root | diamond: main fork | all cells have finite pseudotime',fontsize=6.8)
        fig.text(.12,.012,'Original state labels retained. Branch location remains sensitive to ncenter and cell sampling.',fontsize=6.8)
        fig.canvas.draw()
        box = ax.get_window_extent()
        assert abs(box.width-box.height)<.5
        for text in [*labels,ax.title,*fig.texts,*(ax.get_legend().get_texts() if mode=='legend' else [])]:
            box = text.get_window_extent(fig.canvas.get_renderer())
            assert box.x0>=0 and box.y0>=0 and box.x1<=fig.bbox.width and box.y1<=fig.bbox.height
        stem = f'Monocle3-{view}-{mode}-1'
        # Set the file or directory path.
        fig.savefig(ROOT,dpi=PNG_DPI,facecolor='white',
            metadata={'Description':json.dumps({'parameters':params,'display_colors':palette,
              'label_layout':layout,'anchor_cell':ANCHOR_CELL if view=='cell' else None,
              'input_sha256':saved['input_sha256']})})
        # Set the file or directory path.
        editable_pdf(fig,ROOT)
        figure_checks.append({'view':view,'mode':mode,'cells':len(dots.get_offsets()),'labels':len(labels),
                              'legend_entries':len(handles) if mode=='legend' else 0})
        display(fig)
        plt.close(fig)


In [ ]:
#6. =========== Verify saved figures and paired results
assert result.n_obs==7355 and np.isfinite(result.obs.pseudotime).all()
assert len(nodes)==109 and len(edges)==108 and root_node=='Y_1'
assert set(values)==set(STATE_COLORS) and set(stage)==set(STAGE_COLORS)
assert result.obs.display_state.value_counts()[['Ia candidate','II candidate','Ia/II precursor candidate']].tolist()==[385,71,2249]
assert [row['labels'] for row in figure_checks]==[12,0,9,0]
assert [row['legend_entries'] for row in figure_checks]==[0,12,0,9]
assert all(row['cells']==7355 for row in figure_checks)
for view in ('cell','stage'):
    for mode in ('label','legend'):
        stem = f'Monocle3-{view}-{mode}-1'
        # Set the file or directory path.
        with Image.open(ROOT) as image:
            assert all(abs(dpi-600)<1 for dpi in image.info['dpi'])
            assert image.size==((5160,4980) if mode=='label' else (6720,4620))
        # Set the file or directory path.
        reader = PdfReader(ROOT)
        page = reader.pages[0]
        operations = ContentStream(page['/Contents'],reader).operations
        assert sum(op==b'W' for _,op in operations)==3
        assert not any(op in (b'W*',b'Do') for _,op in operations)
        assert not list(page.images) and 'UMAP1' in page.extract_text()
print('Verified: four views, 600-DPI PNGs, editable vector PDFs, and a matched RDS/H5AD pair.')
shutil.rmtree(TASK_TMP)  # Remove only temporary files created by this notebook.
